In [ ]:
from IPython.display import display, HTML
from dataclasses import fields

import r
import pandas as pd
import algorithm
from render import render_diff, table_results

%matplotlib widget

In [ ]:
algorithm_marching_cubes_lewiner = (
    "Marching Cubes, Lewiner",
    algorithm.MarchingCubes,
    {"method": "lewiner"}
)

algorithm_marhing_cubes_lorensen = (
    "Marching Cubes, Lorensen",
    algorithm.MarchingCubes,
    {"method": "lorensen"}
)

algorithm_flexicubes_default = (
    "FlexiCubes, default",
    algorithm.FlexiCubes,
    {"method": "default"}
)

algorithm_flexicubes_gradient = (
    "FlexiCubes, gradient",
    algorithm.FlexiCubes,
    {"method": "gradient"}
)

algorithm_flexicubes_learn = (
    "FlexiCubes, learn from random",
    algorithm.FlexiCubes,
    {"method": "learn"}
)

algorithms_marching_cubes = [
    algorithm_marching_cubes_lewiner,
    algorithm_marhing_cubes_lorensen
]

algorithms_flexicubes = [
    algorithm_flexicubes_default,
    algorithm_flexicubes_gradient,
    algorithm_flexicubes_learn,
]

all_algorithms = [
    *algorithms_marching_cubes,
    *algorithms_flexicubes
]

In [ ]:
resolution_settings = [
    (f"resolution={r}", {"resolution": r}) for r in [10, 20, 30]
]

large_resolution_settings = [
    (f"resolution={r}", {"resolution": r}) for r in [64, 128, 256]
]

In [ ]:
function_sphere = (
    "Sphere",
    r.Sphere(radius=0.3),
    ((-.3, .3), (-.3, .3), (-.3, .3))
)

function_cube_with_hole = (
    "Cube with hole",
    (
        r.Box(size=(0.2, 0.2, 0.2)) & ~r.CylinderZ(radius=0.1, height=8.0)
    ),
    ((-.2, .2), (-.2, .2), (-.2, .2))
)

function_cube_with_hole_thin_walls = (
    "Cube with hole, thin walls",
    (
        r.Box(size=(0.2, 0.2, 0.2)) & ~r.CylinderZ(radius=0.18, height=8.0)
    ),
    ((-.2, .2), (-.2, .2), (-.2, .2))
)

function_cube_with_hole_close_to_max = (
    "Cube with hole, close to max",
    (
        r.Box(size=(0.49, 0.49, 0.49)) & ~r.CylinderZ(radius=0.1, height=8.0)
    ),
    ((-.49, .49), (-.49, .49), (-.49, .49))
)

In [ ]:
def process_r_function(algorithms, fun, extra_settings, dimensions):
    results = {}
    for title, Algorithm, default_settings in algorithms:
        updated_settings = {**default_settings, **extra_settings}
        display(HTML(f"<h2>Processing: {title}</h2>"))
        algo = Algorithm(updated_settings)
        algo.fit(fun, dimensions)
        render_diff(algo.mesh, fun)
        results[title] = algo
    return results

In [ ]:
def compute_with_settings(experiment_name, algorithms, function_settings, settings):
    results = {}
    fun_name, fun, dimensions = function_settings
    for config_name, config_options in settings:
        heading = f"{fun_name}, {config_name}"
        display(HTML(f"<h1>{heading}</h1>"))
        result = process_r_function(algorithms, fun, config_options, dimensions)
        display(table_results(result))
        results[heading] = result
    return results

In [ ]:
results = compute_with_settings(
    "sphere",
    all_algorithms,
    function_sphere,
    resolution_settings
)

In [ ]:
results = compute_with_settings(
    "cube_with_hole",
    all_algorithms,
    function_cube_with_hole,
    resolution_settings
)

In [ ]:
results = compute_with_settings(
    "thin_walls",
    all_algorithms,
    function_cube_with_hole_thin_walls,
    resolution_settings
)

In [ ]:
results = compute_with_settings(
    "almost_max",
    all_algorithms,
    function_cube_with_hole_close_to_max,
    resolution_settings
)

In [ ]:
results = compute_with_settings(
    "gradient_step",
    [algorithm_flexicubes_gradient],
    function_cube_with_hole_close_to_max,
    [
        (f"gradient_step={gs}", {"resolution": 30, "gradient_step": gs})
        for gs in [1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9]
    ]
)

In [ ]:
results = compute_with_settings(
    "large_resolution",
    [*algorithms_marching_cubes, algorithm_flexicubes_default, algorithm_flexicubes_gradient],
    function_cube_with_hole_close_to_max,
    large_resolution_settings
)